# 조직병리 세포핵 세그멘테이션 — MoNuSeg 2018 불균형 손실 함수 비교

## [사전 조사] SoTA 참고 (rules.md §5-0)

| 항목 | 내용 |
|------|------|
| 데이터셋 | MoNuSeg 2018 Challenge (30 train / 14 test, TCGA) |
| 모달리티 | H&E 염색 병리 슬라이드 (Histopathology, RGB) |
| 태스크 | Binary: Background(0) / Nucleus(1) |
| 불균형 | BG : Nucleus ≈ 3:1 ~ 8:1 (완만한 불균형, 하지만 경계 난이도 높음) |
| SoTA (HoverNet) | Dice ≈ 0.826, AJI ≈ 0.618 |
| SoTA (U-Net) | Dice ≈ 0.790~0.800 |
| **선택 모델** | **U-Net (ResNet34, ImageNet pretrained)** |
| 특이점 | RGB 병리 이미지 (CT/MRI/초음파와 완전히 다른 모달리티) |
| 전처리 | 1000×1000 원본 → 256×256 패치 추출 (overlapping) |

**연구 목적**: 완전히 새로운 모달리티(병리 슬라이드) 추가, 완만한 불균형에서의 LWCE 효과 측정  
**비교 Loss**: `ce_dice`, `wce_dice`, `lwce_dice`, `plwce_dice`, `cb_dice`  
**평가 지표**: Dice, Sensitivity, Specificity, AUC

---

## [project-planner] 실험 계획

| 단계 | 내용 | 완료 기준 |
|------|------|----------|
| 0 | 환경 설정 | device 확인 |
| 1 | 이미지 다운로드 + 패치 추출 + DataLoader | `len(patch_files) > 1000` |
| 2 | 클래스 비율 계산 | `class_counts = [bg, nucleus]` 출력 |
| 3 | 모델 + 유틸리티 함수 | `build_model()` 성공 |
| 4 | 학습 함수 정의 | `train_model()` 정의 |
| 5 | Optuna alpha 탐색 | `best_alpha_plwce` 확보 |
| 6 | 전체 Loss 비교 학습 | `all_results` 딕셔너리 완성 |
| 7 | 시각화 (학습 곡선 + 예측 결과) | PNG 저장 |
| 8 | 최종 평가 지표 + JSON/Excel 저장 | 파일 저장 확인 |

### 데이터 다운로드 안내
- Kaggle: `kagglehub.dataset_download('andrewmvd/monuseg-2018')` 시도
- 또는 공식 사이트: https://monuseg.grand-challenge.org/
- 수동 배치: `/tmp/monuseg_raw/` 하위에 아래 구조:
  ```
  /tmp/monuseg_raw/
    MoNuSeg Training Data/
      Tissue Images/*.tif  (or *.png)
      Annotations/*.png    (이진 마스크 — PNG로 변환된 경우)
    MoNuSegTestData/
      *.tif, *.png
  ```
- **주의**: 원본 데이터는 XML 어노테이션 형식 → 아래 코드에서 자동 변환

### 패치 추출 전략
- 원본 이미지: 1000×1000 (40x 배율)
- 패치 크기: 256×256, Stride: 128 (50% overlap)
- 이미지 단위로 Train/Val 분할 (동일 이미지 패치 간 leakage 방지)

In [ ]:
# ── Cell 0: 환경 설정 + 패키지 설치 ──────────────────────────────────────────
import subprocess, sys

for pkg in ['segmentation-models-pytorch', 'optuna', 'openpyxl',
            'opencv-python', 'lxml', 'tifffile']:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])

import os, warnings, json, random, glob
warnings.filterwarnings('ignore')

import numpy as np
import cv2
import pandas as pd
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
import segmentation_models_pytorch as smp
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

sys.path.insert(0, '/root/imbalanced-data-LWCE/medical_data')
from custom_losses import get_loss_function

# ── Google Drive 마운트 (Colab) ───────────────────────────────────────────────
# MoNuSeg 데이터 Drive 업로드 경로:
#   MyDrive/imbalanced-data-LWCE/monuseg/
#     MoNuSeg Training Data/Tissue Images/*.tif
#     MoNuSeg Training Data/Annotations/*.xml
#     MoNuSegTestData/*.tif, *.xml
GDRIVE_DATA_PATH = '/content/drive/MyDrive/imbalanced-data-LWCE/monuseg'

IS_COLAB = False
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IS_COLAB = True
    print('Google Drive 마운트 완료')
except Exception:
    print('Colab 환경 아님 — 로컬/수동 경로 사용')

# ── 실험 설정 ─────────────────────────────────────────────────────────────────
DOMAIN      = 'monuseg'
NUM_CLASSES = 2
CLASS_NAMES = ['Background', 'Nucleus']
PATCH_SIZE  = 256
PATCH_STRIDE = 128
BATCH_SIZE  = 16
NUM_WORKERS = 4
SEED        = 42

MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
STD  = np.array([0.229, 0.224, 0.225], dtype=np.float32)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

RESULTS_DIR = '/root/imbalanced-data-LWCE/medical_data/results'
os.makedirs(RESULTS_DIR, exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
print('환경 설정 완료')

In [ ]:
# ── Cell 1: 데이터 로드 + 패치 추출 + Dataset + DataLoader ────────────────────
#
# [데이터 준비 방법]
# ── 옵션 A: Google Colab + Google Drive (권장) ────────────────────────────────
#   1. MoNuSeg 2018 데이터를 공식 사이트 또는 아래에서 다운로드:
#      https://monuseg.grand-challenge.org/
#   2. Google Drive에 업로드:
#      MyDrive/imbalanced-data-LWCE/monuseg/
#        MoNuSeg Training Data/Tissue Images/*.tif
#        MoNuSeg Training Data/Annotations/*.xml
#        MoNuSegTestData/*.tif, *.xml
#   3. Cell 0 실행 → 자동으로 /tmp/monuseg_raw/ 에 복사
# ── 옵션 B: 로컬 실행 ─────────────────────────────────────────────────────────
#   /tmp/monuseg_raw/ 에 위 구조로 직접 배치
# ─────────────────────────────────────────────────────────────────────────────

RAW_DIR = '/tmp/monuseg_raw'

# Google Drive → /tmp 복사 (Colab 환경)
tif_check = glob.glob(os.path.join(RAW_DIR, '**', '*.tif'), recursive=True)
if IS_COLAB and os.path.exists(GDRIVE_DATA_PATH) and len(tif_check) < 5:
    import shutil
    print('Google Drive에서 데이터 복사 중... (최초 1회)')
    os.makedirs(RAW_DIR, exist_ok=True)
    shutil.copytree(GDRIVE_DATA_PATH, RAW_DIR, dirs_exist_ok=True)
    tif_check = glob.glob(os.path.join(RAW_DIR, '**', '*.tif'), recursive=True)
    print(f'복사 완료: TIF 파일 {len(tif_check)}개')
elif len(tif_check) >= 5:
    print(f'기존 데이터 사용: {len(tif_check)}개 TIF 파일 발견')
else:
    os.makedirs(RAW_DIR, exist_ok=True)
    print('[데이터 없음] 아래 방법 중 하나를 선택하세요:')
    print('  옵션 A (Colab): Google Drive에 monuseg/ 폴더 업로드 후 Cell 0 재실행')
    print('  옵션 B (로컬):  /tmp/monuseg_raw/ 에 직접 배치:')
    print('    /tmp/monuseg_raw/MoNuSeg Training Data/Tissue Images/*.tif')
    print('    /tmp/monuseg_raw/MoNuSeg Training Data/Annotations/*.xml')

# ── XML 어노테이션 → 이진 마스크 변환 함수 ───────────────────────────────────
def xml_to_mask(xml_path, img_h, img_w):
    """MoNuSeg XML polygon annotation → binary mask (Nucleus=1)"""
    try:
        from lxml import etree
        tree = etree.parse(xml_path)
        root = tree.getroot()
        mask = np.zeros((img_h, img_w), dtype=np.uint8)

        for region in root.findall('.//Region'):
            vertices = region.findall('.//Vertex')
            if len(vertices) < 3:
                continue
            pts = np.array(
                [[float(v.get('X')), float(v.get('Y'))] for v in vertices],
                dtype=np.float32
            ).reshape(-1, 1, 2)
            pts = np.clip(pts, 0, [img_w - 1, img_h - 1]).astype(np.int32)
            cv2.fillPoly(mask, [pts], color=1)

        return mask
    except Exception as e:
        print(f'XML 변환 오류 ({xml_path}): {e}')
        return np.zeros((img_h, img_w), dtype=np.uint8)


# ── 이미지-마스크 쌍 탐색 ─────────────────────────────────────────────────────
tif_files  = glob.glob(os.path.join(RAW_DIR, '**', '*.tif'),  recursive=True)
xml_files  = glob.glob(os.path.join(RAW_DIR, '**', '*.xml'),  recursive=True)
png_masks  = glob.glob(os.path.join(RAW_DIR, '**', '*_mask.png'), recursive=True)

print(f'TIF 이미지: {len(tif_files)}')
print(f'XML 어노테이션: {len(xml_files)}')
print(f'PNG 마스크: {len(png_masks)}')

# ── 패치 추출 (최초 1회) ─────────────────────────────────────────────────────
PATCH_DIR = '/tmp/monuseg_patches'
os.makedirs(PATCH_DIR, exist_ok=True)

existing_patches = glob.glob(os.path.join(PATCH_DIR, '*.npz'))

if len(existing_patches) < 200 and len(tif_files) > 0:
    print(f'패치 추출 중 (크기={PATCH_SIZE}, stride={PATCH_STRIDE})...')

    img_dict   = {os.path.splitext(os.path.basename(f))[0]: f for f in tif_files}
    xml_dict   = {os.path.splitext(os.path.basename(f))[0]: f for f in xml_files}
    pmask_dict = {os.path.splitext(os.path.basename(f))[0].replace('_mask', ''): f
                  for f in png_masks}

    all_pngs = glob.glob(os.path.join(RAW_DIR, '**', '*.png'), recursive=True)
    for pf in all_pngs:
        stem = os.path.splitext(os.path.basename(pf))[0]
        if stem in img_dict and stem not in pmask_dict:
            pmask_dict[stem] = pf

    common_stems = sorted(set(img_dict.keys()) & (set(xml_dict.keys()) | set(pmask_dict.keys())))
    print(f'매칭된 이미지-어노테이션 쌍: {len(common_stems)}')

    patch_count = 0
    for img_idx, stem in enumerate(tqdm(common_stems, desc='Extracting patches')):
        img_path = img_dict[stem]

        img = cv2.imread(img_path)
        if img is None:
            try:
                import tifffile
                img_arr = tifffile.imread(img_path)
                if img_arr.ndim == 2:
                    img = cv2.cvtColor(img_arr, cv2.COLOR_GRAY2BGR)
                elif img_arr.shape[2] == 4:
                    img = cv2.cvtColor(img_arr, cv2.COLOR_RGBA2BGR)
                else:
                    img = img_arr[:, :, :3]
            except Exception:
                print(f'이미지 로드 실패: {img_path}')
                continue

        img_h, img_w = img.shape[:2]
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        if stem in pmask_dict:
            mask = cv2.imread(pmask_dict[stem], cv2.IMREAD_GRAYSCALE)
            mask = (mask > 128).astype(np.uint8) if mask is not None else np.zeros((img_h, img_w), dtype=np.uint8)
        elif stem in xml_dict:
            mask = xml_to_mask(xml_dict[stem], img_h, img_w)
        else:
            continue

        for r in range(0, img_h - PATCH_SIZE + 1, PATCH_STRIDE):
            for c in range(0, img_w - PATCH_SIZE + 1, PATCH_STRIDE):
                img_patch  = img_rgb[r:r+PATCH_SIZE, c:c+PATCH_SIZE]
                mask_patch = mask[r:r+PATCH_SIZE, c:c+PATCH_SIZE]

                # 핵이 전혀 없는 패치 50% 확률로 제거
                if mask_patch.sum() == 0 and random.random() > 0.5:
                    continue

                fname = os.path.join(PATCH_DIR, f'img{img_idx:03d}_r{r:04d}_c{c:04d}.npz')
                np.savez_compressed(fname, image=img_patch.astype(np.uint8),
                                    label=mask_patch.astype(np.int64))
                patch_count += 1

    print(f'패치 추출 완료: {patch_count}개')
elif len(existing_patches) >= 200:
    print(f'캐시 사용: {len(existing_patches)}개 패치 이미 존재')
else:
    print('TIF 파일이 없어 패치 추출을 건너뜁니다. 데이터를 먼저 배치하세요.')

# ── 패치 파일 로드 + 이미지 단위 Train/Val 분할 ───────────────────────────────
patch_files = sorted(glob.glob(os.path.join(PATCH_DIR, '*.npz')))
print(f'총 패치 수: {len(patch_files)}')

img_indices = sorted(set(int(os.path.basename(f).split('_')[0][3:]) for f in patch_files))
tr_img_idx, val_img_idx = train_test_split(img_indices, test_size=0.2, random_state=SEED)
tr_set  = set(tr_img_idx)
val_set = set(val_img_idx)

tr_files  = [f for f in patch_files if int(os.path.basename(f).split('_')[0][3:]) in tr_set]
val_files = [f for f in patch_files if int(os.path.basename(f).split('_')[0][3:]) in val_set]
print(f'Train: {len(tr_files)} patches ({len(tr_img_idx)} images)')
print(f'Val  : {len(val_files)} patches ({len(val_img_idx)} images)')

# ── Dataset 클래스 ─────────────────────────────────────────────────────────────
class MoNuSegDataset(Dataset):
    """
    MoNuSeg Nuclei Pathology Dataset (Patch-based).
    Input:  (3, H, W) — RGB H&E 패치, ImageNet 정규화
    Label:  (H, W)    — 0=Background, 1=Nucleus
    """
    def __init__(self, npz_files, augment=False):
        self.files   = npz_files
        self.augment = augment

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        data  = np.load(self.files[idx])
        image = data['image'].astype(np.uint8)
        label = data['label'].astype(np.int64)

        if self.augment:
            if random.random() > 0.5:
                image = np.fliplr(image).copy()
                label = np.fliplr(label).copy()
            if random.random() > 0.5:
                image = np.flipud(image).copy()
                label = np.flipud(label).copy()
            k = random.randint(0, 3)
            image = np.rot90(image, k).copy()
            label = np.rot90(label, k).copy()

            # H&E 색상 증강
            if random.random() > 0.5:
                alpha = 1.0 + random.uniform(-0.15, 0.15)
                beta  = random.uniform(-10, 10)
                image = np.clip(image.astype(np.float32) * alpha + beta, 0, 255).astype(np.uint8)

        img = image.astype(np.float32) / 255.0
        img = (img - MEAN) / STD
        img = img.transpose(2, 0, 1).astype(np.float32)

        return torch.from_numpy(img), torch.from_numpy(label)


# ── DataLoader ────────────────────────────────────────────────────────────────
train_loader = DataLoader(
    MoNuSegDataset(tr_files,  augment=True),
    batch_size=BATCH_SIZE, shuffle=True,  num_workers=NUM_WORKERS, pin_memory=True
)
val_loader = DataLoader(
    MoNuSegDataset(val_files, augment=False),
    batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True
)
print('DataLoader 구성 완료')

In [ ]:
# ── Cell 2: 클래스 비율 계산 (rules.md §5-3) ──────────────────────────────────
print('클래스 비율 계산 중 (학습 패치 픽셀 단위)...')

class_counts = np.zeros(NUM_CLASSES, dtype=np.int64)
for fp in tqdm(tr_files, desc='Counting pixels'):
    label = np.load(fp)['label'].astype(np.int64)
    class_counts[0] += int((label == 0).sum())
    class_counts[1] += int((label == 1).sum())

class_counts = class_counts.tolist()
total = sum(class_counts)

print()
for c, (name, cnt) in enumerate(zip(CLASS_NAMES, class_counts)):
    print(f'  [{c}] {name:<12}: {cnt:>15,} pixels  ({100 * cnt / total:.2f}%)')

ratio = class_counts[0] / class_counts[1]
print(f'\nBG : Nucleus = {ratio:.1f} : 1  (완만한 불균형 — 경계 난이도가 주요 도전 요소)')
print(f'\nclass_counts = {class_counts}')

In [ ]:
# ── Cell 3: 모델 + 유틸리티 함수 ──────────────────────────────────────────────
#
# [SoTA 참고]
#   HoverNet (Graham et al., 2019): Dice ≈ 0.826, AJI ≈ 0.618
#   DIST (Naylor et al., 2019):     Dice ≈ 0.810
#   U-Net baseline:                 Dice ≈ 0.790~0.800
#   출처: MoNuSeg 2018 Challenge (Kumar et al., 2019, IEEE TMI)
#
# [선택 이유]
#   - RGB H&E 이미지 → ImageNet pretrained 인코더 직접 적용 가능
#   - 완만한 불균형(3:1~8:1)에서 LWCE 효과 = 경계 세밀도 향상 여부
#   - 패치 기반 접근으로 효율적 학습

def to_2ch_logits(p):
    """1채널 logit → 2채널 logit (rules.md §6-2)"""
    return torch.cat([-p, p], dim=1)


def build_model():
    """U-Net (ResNet34, ImageNet pretrained) — Binary nucleus segmentation."""
    return smp.Unet(
        encoder_name    = 'resnet34',
        encoder_weights = 'imagenet',
        in_channels     = 3,
        classes         = 1,
        activation      = None,
    ).to(device)


def compute_val_dice(model, loader):
    """빠른 Val Dice — Optuna 및 학습 모니터링용"""
    model.eval()
    tp = fp = fn = 0
    with torch.no_grad():
        for imgs, masks in loader:
            imgs, masks = imgs.to(device), masks.to(device)
            prob = torch.sigmoid(model(imgs)[:, 0])
            pred = (prob > 0.5).long()
            tp += ((pred == 1) & (masks == 1)).sum().item()
            fp += ((pred == 1) & (masks == 0)).sum().item()
            fn += ((pred == 0) & (masks == 1)).sum().item()
    return float(2 * tp / (2 * tp + fp + fn + 1e-8))


def compute_val_metrics(model, loader):
    """전체 Val 지표: Dice, Sensitivity, Specificity, AUC"""
    model.eval()
    all_probs, all_preds, all_labels = [], [], []
    with torch.no_grad():
        for imgs, masks in loader:
            imgs = imgs.to(device)
            prob = torch.sigmoid(model(imgs)[:, 0]).cpu().numpy()
            pred = (prob > 0.5).astype(np.int64)
            all_probs.append(prob.flatten())
            all_preds.append(pred.flatten())
            all_labels.append(masks.numpy().flatten())

    all_probs  = np.concatenate(all_probs)
    all_preds  = np.concatenate(all_preds)
    all_labels = np.concatenate(all_labels)

    TP = ((all_preds == 1) & (all_labels == 1)).sum()
    FP = ((all_preds == 1) & (all_labels == 0)).sum()
    TN = ((all_preds == 0) & (all_labels == 0)).sum()
    FN = ((all_preds == 0) & (all_labels == 1)).sum()

    dice = 2 * TP / (2 * TP + FP + FN + 1e-8)
    sens = TP / (TP + FN + 1e-8)
    spec = TN / (TN + FP + 1e-8)
    try:
        auc = roc_auc_score(all_labels, all_probs)
    except Exception:
        auc = 0.0

    return {'Dice': float(dice), 'Sensitivity': float(sens),
            'Specificity': float(spec), 'AUC': float(auc)}


test_model = build_model()
n_params   = sum(p.numel() for p in test_model.parameters() if p.requires_grad)
print(f'U-Net (ResNet34) 파라미터 수: {n_params:,}')
del test_model
print('모델 + 유틸리티 함수 준비 완료')

In [ ]:
# ── Cell 4: 학습 함수 ──────────────────────────────────────────────────────────

def train_model(
    loss_name,
    alpha=1.0,
    epochs=50,
    lr=1e-4,
    subset_ratio=1.0,
    tag='',
):
    """
    U-Net(ResNet34) 학습 함수.
    MoNuSeg: 완만한 불균형(3:1~8:1) — 손실 함수 차이가 경계 세밀도에 영향
    """
    model     = build_model()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    criterion = get_loss_function(loss_name, class_counts=class_counts, alpha=alpha)

    name = f'{loss_name}_alpha{alpha:.2f}' if alpha != 1.0 else loss_name
    if tag:
        name = f'{tag}_{name}'

    print(f"\n{'='*60}\nU-Net(ResNet34) + {name}  (epochs={epochs})\n{'='*60}")

    if subset_ratio < 1.0:
        n = max(1, int(len(train_loader.dataset) * subset_ratio))
        sub_ds = torch.utils.data.Subset(
            train_loader.dataset,
            random.sample(range(len(train_loader.dataset)), n)
        )
        loader = DataLoader(sub_ds, batch_size=BATCH_SIZE,
                            shuffle=True, num_workers=NUM_WORKERS)
    else:
        loader = train_loader

    history    = {'loss': [], 'val_dice': []}
    best_dice  = 0.0
    save_path  = f'/tmp/best_unet_monuseg_{name}.pth'

    for epoch in range(epochs):
        model.train()
        epoch_loss = 0.0

        for imgs, masks in tqdm(loader, desc=f'Ep{epoch+1:02d}/{epochs}', leave=False):
            imgs, masks = imgs.to(device), masks.to(device)
            optimizer.zero_grad()
            logits_2ch = to_2ch_logits(model(imgs))
            loss = criterion(logits_2ch, masks)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()

        scheduler.step()
        avg_loss = epoch_loss / len(loader)
        val_dice = compute_val_dice(model, val_loader)

        history['loss'].append(avg_loss)
        history['val_dice'].append(val_dice)

        print(f'Ep{epoch+1:02d} | Loss: {avg_loss:.4f} | Val Dice: {val_dice:.4f}', end='')
        if val_dice > best_dice:
            best_dice = val_dice
            torch.save(model.state_dict(), save_path)
            print('  <- Best!', end='')
        print()

    model.load_state_dict(torch.load(save_path, weights_only=True))
    print(f'최고 Val Dice: {best_dice:.4f}')
    return model, history, best_dice


print('train_model() 함수 준비 완료')

In [ ]:
# ── Cell 5: Optuna alpha 탐색 ─────────────────────────────────────────────────
#
# MoNuSeg 완만한 불균형: 작은 alpha로도 충분할 수 있음
#   plwce: alpha 범위 2.5 ~ 15.0
#   pwce:  alpha 범위 0.2 ~ 2.5

ALPHA_LOW_PLWCE,  ALPHA_HIGH_PLWCE  = 2.5, 15.0
ALPHA_LOW_PWCE,   ALPHA_HIGH_PWCE   = 0.2,  2.5
PROXY_EPOCHS = 5
PROXY_SUBSET = 0.15
N_TRIALS     = 20


def make_objective(loss_name, alpha_low, alpha_high):
    def objective(trial):
        alpha = trial.suggest_float('alpha', alpha_low, alpha_high)
        try:
            _, _, dice = train_model(
                loss_name    = loss_name,
                alpha        = alpha,
                epochs       = PROXY_EPOCHS,
                subset_ratio = PROXY_SUBSET,
                tag          = f'trial{trial.number}',
            )
            return dice
        except Exception as e:
            print(f'Trial {trial.number} 실패: {e}')
            return 0.0
    return objective


print(f'[Optuna] PLWCE alpha 탐색  (범위: {ALPHA_LOW_PLWCE}~{ALPHA_HIGH_PLWCE}, {N_TRIALS} trials)')
study_plwce = optuna.create_study(
    direction  = 'maximize',
    study_name = 'unet_monuseg_plwce_alpha',
    pruner     = optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=2),
)
study_plwce.optimize(make_objective('plwce_dice', ALPHA_LOW_PLWCE, ALPHA_HIGH_PLWCE), n_trials=N_TRIALS)
best_alpha_plwce = study_plwce.best_params['alpha']
print(f'[PLWCE] 최적 alpha = {best_alpha_plwce:.4f}  (Val Dice = {study_plwce.best_value:.4f})')

print(f'\n[Optuna] PWCE alpha 탐색  (범위: {ALPHA_LOW_PWCE}~{ALPHA_HIGH_PWCE}, {N_TRIALS} trials)')
study_pwce = optuna.create_study(
    direction  = 'maximize',
    study_name = 'unet_monuseg_pwce_alpha',
    pruner     = optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=2),
)
study_pwce.optimize(make_objective('pwce_dice', ALPHA_LOW_PWCE, ALPHA_HIGH_PWCE), n_trials=N_TRIALS)
best_alpha_pwce = study_pwce.best_params['alpha']
print(f'[PWCE]  최적 alpha = {best_alpha_pwce:.4f}  (Val Dice = {study_pwce.best_value:.4f})')

optuna_results = {
    'plwce': {
        'best_alpha': best_alpha_plwce,
        'best_proxy_dice': study_plwce.best_value,
        'trials': [{'number': t.number, 'alpha': t.params.get('alpha'), 'value': t.value}
                   for t in study_plwce.trials if t.value is not None],
    },
    'pwce': {
        'best_alpha': best_alpha_pwce,
        'best_proxy_dice': study_pwce.best_value,
        'trials': [{'number': t.number, 'alpha': t.params.get('alpha'), 'value': t.value}
                   for t in study_pwce.trials if t.value is not None],
    },
}
with open(os.path.join(RESULTS_DIR, f'{DOMAIN}_optuna_results.json'), 'w') as f:
    json.dump(optuna_results, f, indent=2, ensure_ascii=False)
print(f'Optuna 결과 저장: {RESULTS_DIR}/{DOMAIN}_optuna_results.json')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, study, sname, a_range in [
    (axes[0], study_plwce, 'PLWCE', f'{ALPHA_LOW_PLWCE}~{ALPHA_HIGH_PLWCE}'),
    (axes[1], study_pwce,  'PWCE',  f'{ALPHA_LOW_PWCE}~{ALPHA_HIGH_PWCE}'),
]:
    trials = [t for t in study.trials if t.value is not None]
    alphas = [t.params['alpha'] for t in trials]
    values = [t.value for t in trials]
    best_a = study.best_params['alpha']
    best_v = study.best_value
    ax.scatter(alphas, values, alpha=0.5, s=40, label='Trials')
    ax.axvline(best_a, color='red', linestyle='--', label=f'Best alpha={best_a:.2f}')
    ax.scatter([best_a], [best_v], color='red', s=100, zorder=5)
    ax.set_xlabel('alpha'); ax.set_ylabel('Val Dice (proxy)')
    ax.set_title(f'{sname} alpha 탐색 (MoNuSeg, 범위 {a_range})')
    ax.legend(); ax.grid(True)

plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, f'{DOMAIN}_optuna_search.png'), dpi=100)
plt.show()

In [ ]:
# ── Cell 6: 전체 Loss 비교 실험 ───────────────────────────────────────────────

FINAL_EPOCHS = 50
FINAL_LR     = 1e-4

try:
    _ = best_alpha_plwce
except NameError:
    try:
        with open(os.path.join(RESULTS_DIR, f'{DOMAIN}_optuna_results.json')) as f:
            d = json.load(f)
        best_alpha_plwce = d['plwce']['best_alpha']
        best_alpha_pwce  = d['pwce']['best_alpha']
        print(f'Optuna 결과 로드: PLWCE alpha={best_alpha_plwce:.4f}')
    except FileNotFoundError:
        best_alpha_plwce = 5.0
        best_alpha_pwce  = 0.5
        print('Optuna 미실행 → 기본값 사용 (PLWCE alpha=5.0)')

experiments = [
    ('ce_dice',    1.0,               'CE+Dice        (기준선)'),
    ('wce_dice',   1.0,               'WCE+Dice'),
    ('lwce_dice',  1.0,               'LWCE+Dice'),
    ('plwce_dice', best_alpha_plwce,  f'PLWCE+Dice     (alpha={best_alpha_plwce:.2f})'),
    ('cb_dice',    1.0,               'CB+Dice'),
]

all_results = {}
for loss_name, alpha, label in experiments:
    model, history, best_dice = train_model(
        loss_name = loss_name,
        alpha     = alpha,
        epochs    = FINAL_EPOCHS,
        lr        = FINAL_LR,
        tag       = 'final',
    )
    all_results[label] = {
        'model':     model,
        'history':   history,
        'best_dice': best_dice,
        'loss_name': loss_name,
        'alpha':     alpha,
    }

print('\n' + '='*50)
print('[MoNuSeg Loss 비교 실험 요약 — Val Dice]')
print(f"{'Loss':<35} {'Best Val Dice':>13}")
print('-' * 50)
for label, v in all_results.items():
    print(f"{label:<35} {v['best_dice']:>13.4f}")

In [ ]:
# ── Cell 7: 시각화 ────────────────────────────────────────────────────────────

COLORS = ['#4878D0', '#EE854A', '#6ACC65', '#D65F5F', '#B47CC7']

# 7-1. 학습 곡선
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
for i, (label, v) in enumerate(all_results.items()):
    h = v['history']
    ax1.plot(h['loss'],     label=label, color=COLORS[i % len(COLORS)])
    ax2.plot(h['val_dice'], label=label, color=COLORS[i % len(COLORS)])

ax1.set_title('Train Loss'); ax1.set_xlabel('Epoch')
ax1.legend(fontsize=7); ax1.grid(True)
ax2.set_title('Val Dice (Nucleus)'); ax2.set_xlabel('Epoch')
ax2.legend(fontsize=7); ax2.grid(True)

plt.suptitle('MoNuSeg — U-Net(ResNet34) 학습 곡선 비교', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, f'{DOMAIN}_training_curves.png'), dpi=100, bbox_inches='tight')
plt.show()
print(f'학습 곡선 저장: {RESULTS_DIR}/{DOMAIN}_training_curves.png')

# 7-2. 예측 결과 시각화 (4열: H&E Patch / GT / Prob Map / Pred)
best_label = max(all_results, key=lambda k: all_results[k]['best_dice'])
best_model = all_results[best_label]['model']
best_model.eval()
print(f'\n시각화 모델: {best_label}  (Val Dice={all_results[best_label]["best_dice"]:.4f})')

val_ds      = MoNuSegDataset(val_files, augment=False)
vis_indices = random.sample(range(len(val_ds)), min(4, len(val_ds)))

fig, axes = plt.subplots(len(vis_indices), 4, figsize=(18, len(vis_indices) * 4))
if len(vis_indices) == 1:
    axes = axes[np.newaxis, :]

for row, idx in enumerate(vis_indices):
    img_t, mask_t = val_ds[idx]
    # 역정규화
    img_vis = img_t.numpy().transpose(1, 2, 0) * STD + MEAN
    img_vis = np.clip(img_vis, 0, 1)

    with torch.no_grad():
        logit = best_model(img_t.unsqueeze(0).to(device))
        prob  = torch.sigmoid(logit[0, 0]).cpu().numpy()
        pred  = (prob > 0.5).astype(np.uint8)

    axes[row, 0].imshow(img_vis)
    axes[row, 0].set_title('H&E Pathology Patch'); axes[row, 0].axis('off')
    axes[row, 1].imshow(mask_t.numpy(), cmap='gray', vmin=0, vmax=1)
    axes[row, 1].set_title('Ground Truth (Nucleus=White)'); axes[row, 1].axis('off')
    axes[row, 2].imshow(prob, cmap='jet', vmin=0, vmax=1)
    axes[row, 2].set_title('Nucleus Probability Map'); axes[row, 2].axis('off')
    axes[row, 3].imshow(pred, cmap='gray', vmin=0, vmax=1)
    axes[row, 3].set_title(f'Prediction ({best_label.split("(")[0].strip()[:12]})')
    axes[row, 3].axis('off')

plt.suptitle(f'MoNuSeg — 예측 결과 시각화 ({best_label})', fontsize=12, y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, f'{DOMAIN}_prediction_vis.png'), dpi=100, bbox_inches='tight')
plt.show()
print(f'예측 결과 저장: {RESULTS_DIR}/{DOMAIN}_prediction_vis.png')

In [ ]:
# ── Cell 8: 최종 정량 평가 + JSON + Excel 저장 ────────────────────────────────

print('\n[전체 모델 종합 평가 — Val Set]')
print(f"{'Loss':<35} {'Dice':>7} {'Sens':>7} {'Spec':>7} {'AUC':>7}")
print('-' * 65)

final_results = {}
for label, v in all_results.items():
    metrics = compute_val_metrics(v['model'], val_loader)
    final_results[label] = {
        'loss_name':     v['loss_name'],
        'alpha':         v['alpha'],
        'best_val_dice': v['best_dice'],
        **metrics,
    }
    print(
        f"{label:<35} "
        f"{metrics['Dice']:>7.4f} "
        f"{metrics['Sensitivity']:>7.4f} "
        f"{metrics['Specificity']:>7.4f} "
        f"{metrics['AUC']:>7.4f}"
    )

# ── 바차트 비교 ───────────────────────────────────────────────────────────────
metric_keys = ['Dice', 'Sensitivity', 'Specificity', 'AUC']
labels_     = list(final_results.keys())

fig, axes = plt.subplots(1, 4, figsize=(22, 5))
for ax, mkey in zip(axes, metric_keys):
    scores = [final_results[lb][mkey] for lb in labels_]
    bars   = ax.bar(range(len(labels_)), scores, color=COLORS[:len(labels_)], alpha=0.85)
    ax.set_xticks(range(len(labels_)))
    ax.set_xticklabels(
        [lb.split('(')[0].strip()[:12] for lb in labels_],
        rotation=30, ha='right', fontsize=8
    )
    ax.set_title(mkey); ax.set_ylim(0, 1.05); ax.grid(axis='y', alpha=0.4)
    best_idx = int(np.argmax(scores))
    bars[best_idx].set_edgecolor('red'); bars[best_idx].set_linewidth(2.5)
    for bar, score in zip(bars, scores):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
                f'{score:.3f}', ha='center', va='bottom', fontsize=7)

plt.suptitle('MoNuSeg — Loss별 최종 평가 지표 비교 (Val Set)', fontsize=13)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, f'{DOMAIN}_final_metrics.png'), dpi=100, bbox_inches='tight')
plt.show()
print(f'평가 차트 저장: {RESULTS_DIR}/{DOMAIN}_final_metrics.png')

# ── JSON 저장 ─────────────────────────────────────────────────────────────────
save_data = {
    'domain':       'MoNuSeg 2018 Multi-organ Nuclei Segmentation',
    'model':        'U-Net (ResNet34, ImageNet pretrained)',
    'sota_ref': {
        'HoverNet': {'Dice': 0.826, 'AJI': 0.618},
        'U-Net':    {'Dice': '0.790~0.800'},
    },
    'num_classes':  NUM_CLASSES,
    'class_counts': {n: int(c) for n, c in zip(CLASS_NAMES, class_counts)},
    'imbalance':    {'BG_Nucleus': round(class_counts[0] / class_counts[1], 1)},
    'patch_size':   PATCH_SIZE,
    'patch_stride': PATCH_STRIDE,
    'train_patches': len(tr_files),
    'val_patches':   len(val_files),
    'final_epochs': FINAL_EPOCHS,
    'results': {
        k: {mk: float(mv) if isinstance(mv, (float, np.floating)) else mv
            for mk, mv in v.items() if mk != 'model'}
        for k, v in final_results.items()
    },
    'best_model': max(final_results, key=lambda k: final_results[k]['Dice']),
}
with open(os.path.join(RESULTS_DIR, f'{DOMAIN}_final_results.json'), 'w', encoding='utf-8') as f:
    json.dump(save_data, f, indent=2, ensure_ascii=False)
print(f'JSON 저장: {RESULTS_DIR}/{DOMAIN}_final_results.json')

# ── Excel 저장 ────────────────────────────────────────────────────────────────
summary_rows = []
for label, v in final_results.items():
    summary_rows.append({
        'Loss_Function':   label,
        'loss_name':       v['loss_name'],
        'alpha':           round(float(v['alpha']), 4),
        'Best_Val_Dice':   round(v['best_val_dice'], 4),
        'Val_Dice':        round(v['Dice'],        4),
        'Val_Sensitivity': round(v['Sensitivity'], 4),
        'Val_Specificity': round(v['Specificity'], 4),
        'Val_AUC':         round(v['AUC'],         4),
        'BG_Nucleus_ratio': round(class_counts[0] / class_counts[1], 1),
        'patch_size':      PATCH_SIZE,
        'patch_stride':    PATCH_STRIDE,
        'epochs':          FINAL_EPOCHS,
        'model':           'U-Net (ResNet34)',
    })
df_summary = pd.DataFrame(summary_rows)

history_rows = []
for label, v in all_results.items():
    for ep, (loss, dice) in enumerate(
        zip(v['history']['loss'], v['history']['val_dice']), 1
    ):
        history_rows.append({
            'Loss_Function': label,
            'Epoch':         ep,
            'Train_Loss':    round(loss, 6),
            'Val_Dice':      round(dice, 6),
        })
df_history = pd.DataFrame(history_rows)

excel_path = os.path.join(RESULTS_DIR, f'{DOMAIN}_final_results.xlsx')
with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:
    df_summary.to_excel(writer, sheet_name='Summary',          index=False)
    df_history.to_excel(writer, sheet_name='Training_History', index=False)
print(f'Excel 저장: {excel_path}')

print(f"\n최고 모델: {save_data['best_model']}")
print(f'BG : Nucleus = {save_data["imbalance"]["BG_Nucleus"]:>5.1f} : 1  (완만한 불균형)')